In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# 🚨 Incident Response Assistant

## What We're Building

An incident response workflow that:
1. **Gathers logs** — collects and parses incident data
2. **Summarizes the incident** — creates a structured incident report
3. **Suggests runbook steps** — recommends remediation actions
4. **Pauses for confirmation** — engineer reviews before execution

## Architecture

```
Incident Alert
      │
      ▼
┌────────────┐    ┌────────────┐    ┌────────────┐    ┌──────────┐
│ Gather     │ ──▶│ Summarize  │ ──▶│ Suggest    │ ──▶│ ⏸ PAUSE  │
│ Logs       │    │ Incident   │    │ Runbook    │    │ Confirm  │
└────────────┘    └────────────┘    └────────────┘    └────┬─────┘
                                                          │
                                                    Engineer approves
                                                          │
                                                     ┌────▼─────┐
                                                     │ Execute  │
                                                     │ + Report │
                                                     └──────────┘
```

## Key LangGraph Concepts
- **HITL with interrupt_before** — pause for engineer review
- **Structured state** — accumulate incident data through nodes

## Stack
- **LangGraph** — workflow with human-in-the-loop
- **Ollama** — local LLM

## Step 1 — Imports & Setup

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from langchain_ollama import ChatOllama
from langchain.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOllama(model="qwen3.5:9b", temperature=0.1)
print("All imports successful!")

## Step 2 — Simulated Log Data & State

In [ ]:
class IncidentState(TypedDict):
    alert_text: str             # Raw alert
    raw_logs: str               # Gathered log entries
    incident_summary: str       # Structured summary
    severity: str               # P1/P2/P3/P4
    suggested_steps: str        # Runbook remediation steps
    engineer_decision: str      # approve / modify / reject
    execution_report: str       # Final action report


# Simulated incident alert + logs
sample_alert = """ALERT: API Gateway returning 502 Bad Gateway
Triggered: 2024-03-15 14:23:00 UTC
Service: api-gateway-prod
Region: us-east-1
Error rate: 47% (threshold: 5%)
Affected endpoints: /api/v2/users, /api/v2/orders, /api/v2/payments
"""

sample_logs = """[14:22:45] api-gateway | upstream connect error: connection refused
[14:22:46] user-service | OOM Kill: container exceeded 2GB memory limit
[14:22:46] user-service | Process exited with code 137 (SIGKILL)
[14:22:47] k8s-scheduler | Pod user-service-7b4d9f crashed. Restart count: 5
[14:22:48] api-gateway | 502 Bad Gateway for GET /api/v2/users (upstream: user-service:8080)
[14:22:50] order-service | Connection to user-service timed out after 30s
[14:22:51] api-gateway | 502 Bad Gateway for POST /api/v2/orders (upstream: order-service:8080)
[14:22:55] payment-service | Failed to validate user token: user-service unavailable
[14:22:56] api-gateway | 502 Bad Gateway for POST /api/v2/payments (upstream: payment-service:8080)
[14:23:00] monitoring | ALERT triggered: api-gateway error rate 47% > 5% threshold
[14:23:01] k8s-scheduler | user-service memory usage trend: 1.2GB → 1.8GB → 2.1GB (last 3 hours)
[14:23:02] deployment-log | Last deploy to user-service: 2024-03-15 11:00 UTC (commit abc123: "Add user analytics caching")
"""

print("Sample incident data loaded")
print(f"Alert: {len(sample_alert)} chars")
print(f"Logs: {len(sample_logs.split(chr(10)))} log lines")

## Step 3 — Define Workflow Nodes

In [ ]:
# --- Node 1: Gather & parse logs ---
def gather_logs(state: IncidentState) -> dict:
    """Simulate gathering logs from multiple sources."""
    print("📋 Node: gather_logs — Collecting logs from monitoring systems...")
    # In production this would query Datadog, CloudWatch, ELK, etc.
    return {"raw_logs": sample_logs}


# --- Node 2: Summarize incident ---
summarize_prompt = ChatPromptTemplate.from_template(
    """You are an SRE analyzing an incident. Create a structured incident summary.

Alert:
{alert}

Logs:
{logs}

Provide:
1. TITLE: One-line incident title
2. SEVERITY: P1 (critical production down) / P2 (major degradation) / P3 (minor) / P4 (low)
3. ROOT CAUSE: What likely caused this (based on log evidence)
4. IMPACT: What's affected and who's impacted
5. TIMELINE: Key events in chronological order
6. BLAST RADIUS: Which services are affected

Incident summary:"""
)


def summarize_incident(state: IncidentState) -> dict:
    print("📊 Node: summarize_incident — Analyzing logs...")
    chain = summarize_prompt | llm | StrOutputParser()
    summary = chain.invoke({"alert": state["alert_text"], "logs": state["raw_logs"]})

    # Extract severity
    severity = "P2"  # default
    for line in summary.split("\n"):
        if "SEVERITY" in line.upper():
            for p in ["P1", "P2", "P3", "P4"]:
                if p in line.upper():
                    severity = p
                    break

    print(f"     Severity: {severity}")
    return {"incident_summary": summary, "severity": severity}


# --- Node 3: Suggest runbook steps ---
runbook_prompt = ChatPromptTemplate.from_template(
    """Based on this incident summary, suggest specific remediation steps.

Incident Summary:
{summary}

Logs:
{logs}

Provide numbered steps an on-call engineer should take.
For each step:
- The action (be specific — include example commands where helpful)
- Why this step helps
- Risk level of this action (safe / caution / risky)

Remediation steps:"""
)


def suggest_runbook(state: IncidentState) -> dict:
    print("📖 Node: suggest_runbook — Generating remediation steps...")
    chain = runbook_prompt | llm | StrOutputParser()
    steps = chain.invoke({"summary": state["incident_summary"], "logs": state["raw_logs"]})
    return {"suggested_steps": steps}


# --- Node 4: Execute and report (after human approval) ---
report_prompt = ChatPromptTemplate.from_template(
    """Generate a final incident response report.

Incident Summary:
{summary}

Approved Remediation Steps:
{steps}

Engineer Decision: {decision}

Write a formal incident report including:
1. Executive summary (2-3 sentences)
2. Actions taken
3. Current status
4. Follow-up items / post-mortem tasks

Incident report:"""
)


def execute_and_report(state: IncidentState) -> dict:
    print("📄 Node: execute_and_report — Generating final report...")
    chain = report_prompt | llm | StrOutputParser()
    report = chain.invoke({
        "summary": state["incident_summary"],
        "steps": state["suggested_steps"],
        "decision": state["engineer_decision"]
    })
    return {"execution_report": report}


print("All incident response nodes defined")

## Step 4 — Build Graph with HITL Pause

In [ ]:
workflow = StateGraph(IncidentState)

workflow.add_node("gather_logs", gather_logs)
workflow.add_node("summarize_incident", summarize_incident)
workflow.add_node("suggest_runbook", suggest_runbook)
workflow.add_node("execute_and_report", execute_and_report)

workflow.set_entry_point("gather_logs")
workflow.add_edge("gather_logs", "summarize_incident")
workflow.add_edge("summarize_incident", "suggest_runbook")
workflow.add_edge("suggest_runbook", "execute_and_report")
workflow.add_edge("execute_and_report", END)

# Compile with HITL: pause before executing remediation
memory = MemorySaver()
app = workflow.compile(
    checkpointer=memory,
    interrupt_before=["execute_and_report"]  # Engineer must approve first
)

print("Incident response graph compiled (pauses before execution)")

## Step 5 — Run Phase 1: Analyze & Suggest

In [ ]:
config = {"configurable": {"thread_id": "incident-001"}}

initial_state = {
    "alert_text": sample_alert,
    "raw_logs": "",
    "incident_summary": "",
    "severity": "",
    "suggested_steps": "",
    "engineer_decision": "",
    "execution_report": "",
}

print("🚀 Starting incident response workflow...\n")
result = app.invoke(initial_state, config)

print("\n" + "="*60)
print("⏸️ PAUSED — Waiting for engineer approval")
print("="*60)

print("\n📊 Incident Summary:")
print(result["incident_summary"])

print("\n📖 Suggested Remediation Steps:")
print(result["suggested_steps"])

## Step 6 — Engineer Reviews & Approves

In [ ]:
# Simulate engineer decision
engineer_decision = """
APPROVED with modifications:
- Step 1 (rollback): APPROVED — rollback commit abc123 immediately
- Step 2 (increase memory): APPROVED — increase to 4GB, not 3GB
- Step 3 (restart pods): APPROVED — rolling restart, not full restart
- Additional: Page the on-call team lead (this is a P1 during business hours)
- Additional: Open a Jira ticket for post-mortem within 24 hours
"""

# Inject engineer decision and resume
app.update_state(config, {"engineer_decision": engineer_decision})

print("🔄 Resuming with engineer approval...\n")
final_result = app.invoke(None, config)

print("\n" + "="*60)
print("✅ INCIDENT RESPONSE COMPLETE")
print("="*60)
print(final_result["execution_report"])

## 🧠 Key Concepts Recap

| Concept | Implementation |
|---------|---------------|
| **Log gathering** | Node simulates collecting from monitoring tools |
| **LLM analysis** | Structured incident summary from unstructured logs |
| **Runbook generation** | Context-specific remediation steps |
| **HITL gate** | `interrupt_before` pauses for engineer approval |
| **State injection** | `update_state()` adds engineer's decision |

## Production Integrations

| Node | Real Integration |
|------|------------------|
| gather_logs | Datadog, CloudWatch, ELK, Grafana APIs |
| summarize | Could include historical incident matching |
| suggest_runbook | Could query internal runbook database |
| execute | Could run actual kubectl, AWS CLI commands |

## 🔧 Extensions

- **Auto-execute safe steps**: Only pause for risky actions
- **Severity routing**: P1 pages immediately, P4 auto-handles
- **Post-mortem generator**: Auto-create post-mortem doc from incident data
- **Slack integration**: Send updates to incident channel